# Setup

In [1]:
# Setup
import json
import os
import sys
from datetime import datetime, timedelta
import time
from dotenv import load_dotenv, set_key
import random
from pathlib import Path

# hashing for signing
import hashlib
import hmac

# requests
import requests

# data munging
import pandas as pd
import numpy as np

# helper functions
from tiktok_api_helpers import *

## API Setup

In [2]:
new_access_token = False
use_refresh_token = False

In [3]:
if (new_access_token | use_refresh_token):

    # get new acces_token
    token_url = "https://auth.tiktok-shops.com/api/v2/token/get"

    if use_refresh_token:
        auth_code = os.environ.get("TIKTOK_REFRESH_TOKEN")
        
    params = {
        "app_key": app_key,
        "app_secret": app_secret,
        "auth_code": auth_code,
        "grant_type": "authorized_code",
    }
    
    response = requests.get(token_url, params=params, timeout=15)
    data = response.json()['data']
    access_token = data['access_token']
    print(data)

    # save tokens in .env file
    set_key(".env", "TIKTOK_ACCESS_TOKEN", data['access_token'])
    set_key(".env", "TIKTOK_REFRESH_TOKEN", data['refresh_token'])

else:
    access_token = os.environ.get("TIKTOK_ACCESS_TOKEN")
    refresh_token = os.environ.get("TIKTOK_REFRESH_TOKEN")

# Extract Creator Open IDs

In [4]:
# load all creators
df_creators_list = pd.read_excel("all_creators_sorted.xlsx", sheet_name="LIST_CREATOR", usecols=[1, 2, 25])
list_all_creators = df_creators_list['username'].tolist()

In [5]:
# Load or initialize the manifest (tracks which handles are already found)
manifest = pd.read_csv(MANIFEST_CSV)
df_creators = pd.read_csv(CONSOLIDATED_CSV)

# recheck manifest in case of failed runs
manifest["found"] = manifest["handle"].isin(set(df_creators['username']))
manifest.to_csv(MANIFEST_CSV, index=False)

# check handles to find
handles_to_find = manifest.loc[~manifest["found"], "handle"].tolist()
print(f"{len(handles_to_find)} handles left to find out of {len(manifest)} total.\n")

21598 handles left to find out of 21598 total.



In [7]:
manifest['found'].sum()

np.int64(0)

In [7]:
# finish top 2k handles first
top_n = 2000
handles_to_find = handles_to_find[:top_n]

In [8]:
len(handles_to_find)

2000

## First-pass: Chunk size = 5

In [ ]:
# Phase 1: chunk_size=5, single pass through everything not yet found
still_not_found, found_usernames, df_creators = run_pass(handles_to_find, chunk_size=5, df_creators=df_creators)

manifest["found"] = manifest["handle"].isin(found_usernames)
manifest.to_csv(MANIFEST_CSV, index=False)
print(f"\nPhase 1 done. {len(still_not_found)} handles still not found.\n")

[chunk_size=5] 1/400: searching ['ginoroqueiv', 'agapearliyshop', 'rosmar.acaiberry.legit', 'shoppersvirtual', 'reiseyeee_06']


## Second-pass: Chunk size = 1

In [ ]:
# Phase 2: chunk_size=1, only for leftovers from phase 1
if still_not_found:
    still_not_found, found_usernames, df_creators = run_pass(still_not_found, chunk_size=1, df_creators=df_creators)

    found_usernames = set(df_creators["username"])
    manifest["found"] = manifest["handle"].isin(found_usernames)
    manifest.to_csv(MANIFEST_CSV, index=False)

print(f"\nDone. {int(manifest['found'].sum())} / {len(manifest)} handles found.")

In [ ]:
still_not_found